In [2]:
import networkx as nx
import itertools
from math import comb
import matplotlib.pyplot as plt
from typing import List, Tuple, Set

In [2]:
def hamming_distance(a: int, b: int) -> int:
    """Distancia de Hamming entre dos enteros (popcount del XOR)."""
    return (a ^ b).bit_count()

def build_hamming_graph(n: int, d: int) -> nx.Graph:
    """
    Construye el grafo de Hamming H(n, d):
    - Vértices: números enteros de 0 a 2^n - 1 (representan palabras binarias)
    - Arista entre u y v si hamming_distance(u, v) >= d
    """
    num_vertices = 1 << n
    G = nx.Graph()
    G.add_nodes_from(range(num_vertices))
    for u in range(num_vertices):
        for v in range(u + 1, num_vertices):
            if hamming_distance(u, v) >= d:
                G.add_edge(u, v)
    return G


In [3]:
def bron_kerbosch_max_cliques(G: nx.Graph):
    """
    Implementación del algoritmo de Bron–Kerbosch (versión con pivote)
    que encuentra todos los cliques maximales en un grafo no dirigido.
    Adaptado del artículo original "Algorithm 457: Finding All Cliques of an Undirected Graph".
    Retorna una lista de cliques (cada clique es un conjunto de vértices).
    """
    # Convertir los vértices a enteros 0..N-1 (ya lo son)
    vertices = list(G.nodes())
    # Precalcular vecinos como lista de conjuntos para acceso rápido
    neighbors = {v: set(G.neighbors(v)) for v in vertices}
    N = len(vertices)
    # Orden inicial: todos los vértices en "candidates", "not" vacío
    all_vertices = vertices[:]  # lista de todos los vértices en orden
    compsub = []  # R, clique en construcción
    cliques = []  # almacenará los cliques maximales encontrados

    def extend(old, ne, ce):
        """
        old: lista de vértices (primero 'not' (0..ne-1), luego 'candidates' (ne..ce-1))
        ne: número de vértices en not (0 <= ne <= ce)
        ce: número total de elementos en old (len(old))
        """
        nonlocal cliques
        # Paso 1: elegir punto fijo (fixp) con mínimo número de disconexiones
        minnod = ce
        fixp = None
        s = -1
        nod = 0  # indicador si el punto fijo se tomó de candidates (1) o not (0)
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            count = 0
            # Contar disconexiones con el resto de candidatos (desde ne hasta ce-1)
            j = ne
            pos = -1
            while j < ce and count <= minnod:
                if p not in neighbors[old[j]]:  # disconexión
                    count += 1
                    pos = j
                j += 1
            if count < minnod:
                fixp = p
                minnod = count
                if i < ne:
                    s = pos
                else:
                    s = i
                    nod = 1
            i += 1

        # Bucle principal de backtracking
        # nod iterará desde minnod + nod hasta 1
        for _ in range(minnod + nod, 0, -1):
            # Intercambiar el candidato seleccionado (old[s]) con old[ne]
            p = old[s]
            old[s] = old[ne]
            sel = old[ne]
            old[ne] = p

            # Construir nuevo conjunto 'not' (new) y 'candidates' (newcand)
            new = [0] * ce  # preasignamos tamaño máximo
            newne = 0
            # Copiar vértices de 'not' que son vecinos de sel
            for i in range(ne):
                if sel in neighbors[old[i]]:
                    new[newne] = old[i]
                    newne += 1
            newce = newne
            # Copiar vértices de 'candidates' (desde ne+1 hasta ce-1) que son vecinos de sel
            # Nota: el índice ne ya contiene sel, que fue movido; lo saltamos
            for i in range(ne + 1, ce):
                if sel in neighbors[old[i]]:
                    new[newce] = old[i]
                    newce += 1

            compsub.append(sel)

            if newce == 0:
                # Se encontró un clique maximal
                cliques.append(compsub.copy())
            else:
                if newne < newce:
                    # Llamada recursiva con el nuevo conjunto
                    # new[:newce] contiene (not + candidates)
                    extend(new, newne, newce)

            compsub.pop()
            # Mover sel al conjunto 'not' para futuras iteraciones
            ne += 1

            # Si aún quedan candidatos por procesar (nod > 1), seleccionar el siguiente
            # candidato desconectado del punto fijo
            if nod > 1:
                # Buscar siguiente candidato (pos > s) que esté desconectado de fixp
                s = ne
                while s < ce and fixp in neighbors[old[s]]:
                    s += 1
                if s >= ce:
                    break
                # El siguiente candidato ya está en old[s], listo para el siguiente ciclo
            else:
                # Solo un candidato, terminar
                break

    # Iniciar llamada con todos los vértices en candidates, not vacío
    extend(all_vertices, 0, N)
    return cliques


In [6]:
G= build_hamming_graph(8,4)
cliques = bron_kerbosch_max_cliques(G)
cliques

[[0, 15, 51, 60, 85, 90, 102, 105, 170, 204, 240, 255]]

In [4]:
def main():
    # Parámetros de prueba
    n = 8      # longitud de las palabras
    d = 4      # distancia mínima requerida
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")

    print("\n=== Cliques maximales encontrados por nuestra implementación ===")
    our_cliques = bron_kerbosch_max_cliques(G)
    print(f"Número de cliques maximales: {len(our_cliques)}")
    max_size = max(len(c) for c in our_cliques) if our_cliques else 0
    print(f"Tamaño del clique máximo (A({n},{d})): {max_size}")
    # Mostrar primeros 5 cliques como ejemplo
    print("Ejemplo de cliques (primeros 5):")
    for i, clique in enumerate(our_cliques[:5]):
        # Convertir enteros a representación binaria para mejor visualización
        bin_repr = [format(v, f'0{n}b') for v in clique]
        print(f"  {i+1}: {bin_repr}")

    # Comparación con networkx.find_cliques (Bron–Kerbosch implementado en C)
    print("\n=== Comparación con networkx.find_cliques ===")
    nx_cliques = list(nx.find_cliques(G))
    print(f"networkx encontró {len(nx_cliques)} cliques maximales.")
    nx_max_size = max(len(c) for c in nx_cliques) if nx_cliques else 0
    print(f"Tamaño del clique máximo según networkx: {nx_max_size}")

    # Verificar que nuestros cliques coinciden (como conjuntos)
    our_sets = [set(c) for c in our_cliques]
    nx_sets = [set(c) for c in nx_cliques]
    if set(frozenset(s) for s in our_sets) == set(frozenset(s) for s in nx_sets):
        print("¡Los conjuntos de cliques maximales coinciden perfectamente!")
    else:
        print("Advertencia: los conjuntos difieren. Revisar implementación.")

if __name__ == "__main__":
    main()

Construyendo grafo de Hamming H(8,4)...
Vértices: 256, Aristas: 20864

=== Cliques maximales encontrados por nuestra implementación ===
Número de cliques maximales: 1
Tamaño del clique máximo (A(8,4)): 12
Ejemplo de cliques (primeros 5):
  1: ['00000000', '00001111', '00110011', '00111100', '01010101', '01011010', '01100110', '01101001', '10101010', '11001100', '11110000', '11111111']

=== Comparación con networkx.find_cliques ===


KeyboardInterrupt: 

In [3]:
def bron_kerbosch_version2(graph: nx.Graph) -> List[Set[int]]:
    """
    Implementación del algoritmo de Bron–Kerbosch versión 2 (con pivote)
    para encontrar todas las cliques maximales.
    Sigue la lógica del artículo "Algorithm 457" (Comm. ACM, 1973).
    """
    # Convertir el grafo a una matriz de adyacencia booleana para acceso rápido
    nodes = list(graph.nodes())
    index_of = {node: i for i, node in enumerate(nodes)}
    n_nodes = len(nodes)
    adj = [[False]*n_nodes for _ in range(n_nodes)]
    for u, v in graph.edges():
        i, j = index_of[u], index_of[v]
        adj[i][j] = adj[j][i] = True
    # Para simplificar, trabajamos con índices enteros 0..n_nodes-1
    # La función recursiva interna usará listas de enteros (índices)
    
    all_cliques = []   # almacenará las cliques encontradas (como conjuntos de nodos originales)
    
    # Algoritmo recursivo: extiende una clique parcial (compsub)
    # old: array de enteros ordenado [not | candidates] (ne = tamaño de not, ce = tamaño total)
    # ne, ce: índices que separan not y candidates en el arreglo old[0:ne] es not, old[ne:ce] es candidates
    def extend(old: List[int], ne: int, ce: int):
        # old tiene longitud ce, los primeros ne son "not", los siguientes ce-ne son "candidates"
        # Se trabaja in-place, pero creamos un nuevo arreglo new para la llamada recursiva.
        
        # Determinar el punto fijo (fixp) y el candidato con mínimo número de desconexiones
        # (versión 2 del artículo)
        minnod = ce
        fixp = -1
        s = -1           # posición del candidato que se usará como pivote
        nod = 0          # bandera: 1 si fixp viene de candidates
        i = 0
        while i < ce and minnod != 0:
            p = old[i]
            # Contar cuántos candidatos (en la parte candidates) NO son adyacentes a p
            cnt = 0
            j = ne
            pos = -1
            while j < ce and cnt <= minnod:
                if not adj[p][old[j]]:
                    cnt += 1
                    pos = j   # posición de ese candidato desconectado
                j += 1
            if cnt < minnod:
                fixp = p
                minnod = cnt
                if i < ne:
                    # fixp viene de not
                    s = pos
                else:
                    # fixp viene de candidates
                    s = i
                    nod = 1
            i += 1
        
        # Ciclo de backtracking: se repite para cada candidato elegido
        # nod = minnod + nod (en el artículo se itera desde minnod+nod hasta 1)
        # Aquí implementamos la lógica de selección del candidato a mover
        for _ in range(minnod + nod, 0, -1):
            # Seleccionar el candidato en posición s, intercambiarlo con old[ne] (el primero de candidates)
            p = old[s]
            # Intercambio
            old[s], old[ne] = old[ne], p
            sel = old[ne]
            # Construir nuevos conjuntos new (candidatos y not) basados en sel
            new = [0] * ce
            newne = 0
            # Primero los que estaban en not y son adyacentes a sel
            for i in range(ne):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # Luego los que estaban en candidates (old[ne+1:ce]) y son adyacentes a sel
            newce = newne
            for i in range(ne+1, ce):
                if adj[sel][old[i]]:
                    new[newne] = old[i]
                    newne += 1
            # newce es el tamaño de la nueva sección candidates (new[newce:newne])
            # En el algoritmo original: newce = newne (antes de añadir candidates)
            # Pero luego se añaden los candidatos, entonces newne es el total, y newce es el inicio de candidates en new
            # En nuestro new, los primeros newce son los nuevos "not", los siguientes son candidates.
            # Notar que newce es el número de elementos que pasaron de not (adyacentes). Los otros son los nuevos candidates.
            # Ahora se añade sel a compsub
            compsub.append(sel)
            if newne == newce:   # es decir, no hay candidatos nuevos
                # Clique maximal encontrada
                clique_nodes = [nodes[v] for v in compsub]
                all_cliques.append(set(clique_nodes))
            else:
                # Llamada recursiva
                extend(new, newce, newne)
            # Quitar sel de compsub
            compsub.pop()
            # Mover sel a not (incrementar ne)
            ne += 1
            # Si todavía quedan candidatos por procesar (nod > 1 en el artículo)
            if _ > 1:
                # Buscar el siguiente candidato que NO está conectado a fixp
                # (para reducir ramas)
                s = ne
                # Avanzar hasta encontrar un candidato desconectado de fixp
                while s < ce and adj[fixp][old[s]]:
                    s += 1
                if s >= ce:
                    break   # no hay más candidatos que cumplan la condición
        # fin del ciclo for
    
    # Inicialización: ALL contiene todos los nodos (índices)
    all_indices = list(range(n_nodes))
    compsub = []    # pila para la clique actual
    # Llamada inicial: old = all_indices, ne = 0, ce = n_nodes
    extend(all_indices, 0, n_nodes)
    return all_cliques

In [3]:
def bron_kerbosch_version2c(graph: nx.Graph) -> List[Set[int]]:
    """
    Implementación del algoritmo de Bron-Kerbosch versión 2 (con pivote)
    para encontrar todas las cliques maximales en un grafo no dirigido.

    Sigue la lógica del artículo "Algorithm 457" (Comm. ACM, 1973).

    Estrategia central:
      - Se mantiene un arreglo `window` que divide en dos secciones:
          window[0 : num_excluded]          → nodos "excluidos" (ya procesados)
          window[num_excluded : window_size] → nodos "candidatos" (aún por explorar)
      - En cada paso se elige un pivote (el nodo con menos candidatos no-adyacentes)
        para reducir el número de ramas recursivas.
      - `current_clique` es una pila compartida que representa la clique en construcción.

    Complejidad: O(3^(n/3)) en el peor caso, pero el pivote lo acelera en la práctica.
    """

    # ── 1. Construir matriz de adyacencia booleana ──────────────────────────────
    # Permite verificar adyacencia en O(1), lo que acelera el conteo del pivote.

    all_nodes = list(graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(all_nodes)}
    total_nodes = len(all_nodes)

    # adj[i][j] == True  ↔  existe arista entre el nodo i y el nodo j
    adj = [[False] * total_nodes for _ in range(total_nodes)]
    for u, v in graph.edges():
        i, j = node_to_index[u], node_to_index[v]
        adj[i][j] = adj[j][i] = True   # grafo no dirigido: la relación es simétrica

    # ── 2. Resultado acumulado y pila de la clique en construcción ──────────────
    found_cliques: List[Set] = []
    current_clique: List[int] = []   # índices de los nodos en la clique parcial actual

    # ── 3. Función recursiva principal ─────────────────────────────────────────
    def extend(window: List[int], num_excluded: int, window_size: int) -> None:
        """
        Extiende la clique parcial `current_clique` usando los nodos en `window`.

        Parámetros
        ----------
        window       : lista de índices de nodos. Tiene dos secciones:
                         [0 : num_excluded]    → excluidos (no pueden ampliar la clique)
                         [num_excluded : window_size] → candidatos (podrían ampliarla)
        num_excluded : separador entre excluidos y candidatos dentro de `window`.
        window_size  : longitud efectiva de `window` (puede ser < len(window)).
        """

        # ── 3a. Elegir el pivote ────────────────────────────────────────────────
        # El pivote es el nodo (excluido o candidato) con el menor número de
        # candidatos que NO son adyacentes a él.  Esto minimiza las ramas del árbol.

        min_disconnected = window_size   # cota superior inicial (peor caso)
        pivot_node = -1
        pivot_pos_in_window = -1         # posición del primer candidato no-adyacente al pivote
        pivot_is_candidate = False       # True si el pivote viene de la sección candidatos

        for i in range(window_size):
            if min_disconnected == 0:
                break   # ya no se puede mejorar; salir pronto

            node = window[i]

            # Contar candidatos no adyacentes a `node`
            disconnected_count = 0
            last_disconnected_pos = -1

            for j in range(num_excluded, window_size):
                if not adj[node][window[j]]:
                    disconnected_count += 1
                    last_disconnected_pos = j
                    if disconnected_count > min_disconnected:
                        break   # ya superó el mínimo; no hay ganancia

            if disconnected_count < min_disconnected:
                pivot_node = node
                min_disconnected = disconnected_count
                pivot_pos_in_window = last_disconnected_pos

                if i < num_excluded:
                    # El pivote viene de la sección "excluidos":
                    # usamos la posición del candidato desconectado directamente.
                    pass
                else:
                    # El pivote viene de la sección "candidatos":
                    # su propia posición (i) es el primer candidato a probar.
                    pivot_pos_in_window = i
                    pivot_is_candidate = True

        # ── 3b. Ciclo de backtracking ───────────────────────────────────────────
        # Se itera una vez por cada candidato no-adyacente al pivote
        # (más una iteración extra si el propio pivote es candidato).
        num_iterations = min_disconnected + (1 if pivot_is_candidate else 0)

        for iteration in range(num_iterations, 0, -1):

            # Seleccionar el candidato a añadir en esta iteración:
            # intercambiarlo con el primer candidato del window (posición num_excluded).
            selected_node = window[pivot_pos_in_window]
            window[pivot_pos_in_window], window[num_excluded] = (
                window[num_excluded],
                selected_node,
            )
            selected_node = window[num_excluded]   # confirmar tras el swap

            # ── Construir el sub-window para la llamada recursiva ───────────────
            # Solo entran los nodos adyacentes a `selected_node`.
            sub_window: List[int] = []
            sub_num_excluded = 0

            # Nodos excluidos adyacentes a selected_node → siguen siendo excluidos
            for i in range(num_excluded):
                if adj[selected_node][window[i]]:
                    sub_window.append(window[i])
                    sub_num_excluded += 1

            # Candidatos adyacentes a selected_node → pasan a ser candidatos del sub-window
            for i in range(num_excluded + 1, window_size):
                if adj[selected_node][window[i]]:
                    sub_window.append(window[i])

            sub_window_size = len(sub_window)

            # ── Añadir selected_node a la clique en construcción ────────────────
            current_clique.append(selected_node)

            if sub_window_size == sub_num_excluded:
                # No hay candidatos en el sub-window: clique maximal encontrada.
                clique_nodes = {all_nodes[v] for v in current_clique}
                found_cliques.append(clique_nodes)
            else:
                # Aún hay candidatos: continuar expandiendo.
                extend(sub_window, sub_num_excluded, sub_window_size)

            # ── Retroceder: quitar selected_node de la clique ───────────────────
            current_clique.pop()

            # Mover selected_node a la sección "excluidos" para futuras iteraciones
            num_excluded += 1

            # Buscar el siguiente candidato no adyacente al pivote
            if iteration > 1:
                pivot_pos_in_window = num_excluded
                while (
                    pivot_pos_in_window < window_size
                    and adj[pivot_node][window[pivot_pos_in_window]]
                ):
                    pivot_pos_in_window += 1

                if pivot_pos_in_window >= window_size:
                    break   # no quedan candidatos válidos

    # ── 4. Llamada inicial ──────────────────────────────────────────────────────
    # Al inicio: ningún nodo está excluido (num_excluded = 0),
    # todos son candidatos.
    initial_window = list(range(total_nodes))
    extend(initial_window, num_excluded=0, window_size=total_nodes)

    return found_cliques

In [1]:
# ============== Ejemplo de uso ==============
if __name__ == "__main__":
    n = 6  # longitud de las palabras
    d = 4  # distancia mínima deseada
    print(f"Construyendo grafo de Hamming H({n},{d})...")
    G = build_hamming_graph(n, d)
    print(f"Vértices: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")
    
    print("Buscando todas las cliques maximales con Bron-Kerbosch v2...")
    cliques = bron_kerbosch_version2c(G)
    max_size = max(len(c) for c in cliques) if cliques else 0
    print(f"Número de cliques maximales encontradas: {len(cliques)}")
    print(f"Tamaño de la clique máxima: {max_size}")
    print(f"Por lo tanto, A({n},{d}) = {max_size}")
    
    # Mostrar algunas cliques grandes como ejemplo
    print("\nEjemplos de cliques maximales (hasta 5):")
    for i, clique in enumerate(cliques[:5]):
        print(f"  Clique {i+1}: tamaño {len(clique)} -> {clique}")

Construyendo grafo de Hamming H(6,4)...


NameError: name 'build_hamming_graph' is not defined

Nuevo codigo

In [6]:
def bron_kerbosch_max_clique2(adj_mask: list[int], N: int) -> int:
    """
    Algoritmo de Bron-Kerbosch con pivote y coloreo greedy para encontrar
    el tamaño de la clique máxima en el grafo representado por adj_mask.

    Parámetros
    ----------
    adj_mask : lista de N enteros.
               adj_mask[i] es un bitset donde el bit j está activo
               si existe arista entre el vértice i y el vértice j.
    N        : número de vértices del grafo (debe ser <= 63 para eficiencia,
               aunque Python soporta enteros arbitrariamente grandes).

    Retorna
    -------
    Tamaño (número de vértices) de la clique máxima encontrada.

    Estrategia
    ----------
    Los conjuntos se representan como enteros (bitsets):
      - P (candidatos)  : vértices que aún pueden extender la clique actual.
      - X (excluidos)   : vértices ya procesados (garantizan maximalidad).
      - r_size          : tamaño de la clique en construcción (reemplaza al
                          conjunto R, ya que solo necesitamos su cardinalidad).

    Podas aplicadas:
      1. Tamaño:   si r_size + |P| <= max_size, esta rama no puede mejorar.
      2. Coloreo:  si r_size + colores_greedy(P) <= max_size, ídem.
      3. Pivote:   se elige el vértice u en P∪X con mayor |P ∩ N(u)|,
                   reduciendo los candidatos a explorar a P \ N(u).
    """

    # Máscara de N bits para evitar bits "fantasma" al aplicar complemento (~)
    # En Python los enteros son de precisión arbitraria y con signo,
    # por lo que ~x activa infinitos bits superiores si no se enmascara.
    full_mask = (1 << N) - 1

    max_clique_size = 0   # mejor resultado encontrado hasta ahora

    # ── Coloreo greedy ──────────────────────────────────────────────────────────
    def greedy_color_bound(candidates_mask: int) -> int:
        """
        Devuelve una cota superior del tamaño de clique dentro de `candidates_mask`
        mediante coloreo greedy por clases de color independientes.

        Lógica: el tamaño de la clique máxima <= número cromático del grafo.
        Se construyen clases de color (conjuntos independientes) de forma greedy:
        cada vértice se asigna a la primera clase que no tenga ningún vecino suyo.
        El número de clases necesarias es la cota.
        """
        color_classes: list[int] = []   # cada elemento es un bitset (clase de color)
        remaining = candidates_mask

        while remaining:
            # Tomar el vértice de menor índice aún sin colorear
            vertex_bit = remaining & -remaining
            vertex = vertex_bit.bit_length() - 1

            # Buscar la primera clase existente sin vecinos de `vertex`
            placed = False
            for idx, color_class in enumerate(color_classes):
                if not (color_class & adj_mask[vertex]):
                    # Ningún nodo de esta clase es vecino de vertex → asignar aquí
                    color_classes[idx] |= vertex_bit
                    placed = True
                    break

            if not placed:
                # Ninguna clase sirve → abrir una nueva
                color_classes.append(vertex_bit)

            remaining &= ~vertex_bit

        return len(color_classes)

    # ── Expansión recursiva ─────────────────────────────────────────────────────
    def expand(r_size: int, P: int, X: int) -> None:
        """
        Extiende la clique actual (de tamaño r_size) probando cada candidato en P.

        Parámetros
        ----------
        r_size : número de vértices en la clique que se está construyendo.
        P      : bitset de candidatos que pueden ampliar la clique.
        X      : bitset de excluidos (ya procesados en ramas anteriores).
        """
        nonlocal max_clique_size

        # ── Caso base: clique maximal ───────────────────────────────────────────
        if P == 0 and X == 0:
            if r_size > max_clique_size:
                max_clique_size = r_size
            return

        # ── Poda 1: tamaño ─────────────────────────────────────────────────────
        # Si incluso añadiendo todos los candidatos no superamos el máximo, podar.
        if r_size + P.bit_count() <= max_clique_size:
            return

        # ── Poda 2: coloreo greedy ─────────────────────────────────────────────
        # Cota más ajustada: número cromático de P es cota del tamaño de clique en P.
        if r_size + greedy_color_bound(P) <= max_clique_size:
            return

        # ── Elegir pivote u en P ∪ X ───────────────────────────────────────────
        # Criterio: maximizar |P ∩ N(u)| para minimizar los candidatos a explorar.
        # Cuantos más candidatos cubre el pivote, menos ramas se abren.
        union_PX = P | X
        best_pivot = -1
        best_coverage = -1

        temp = union_PX
        while temp:
            u_bit = temp & -temp
            u = u_bit.bit_length() - 1
            coverage = (P & adj_mask[u]).bit_count()
            if coverage > best_coverage:
                best_coverage = coverage
                best_pivot = u
            temp ^= u_bit   # eliminar u_bit de temp

        # Candidatos a explorar: vértices de P que NO son vecinos del pivote.
        # El pivote ya "cubre" sus vecinos, así que no hay que expandirlos desde aquí.
        candidates = P & (~adj_mask[best_pivot] & full_mask)

        # ── Ciclo de backtracking ───────────────────────────────────────────────
        while candidates:
            # Tomar el candidato de menor índice
            v_bit = candidates & -candidates
            v = v_bit.bit_length() - 1

            # Llamada recursiva: añadir v a la clique,
            # restringir P y X a los vecinos de v.
            expand(
                r_size + 1,
                P & adj_mask[v],
                X & adj_mask[v],
            )

            # Mover v de P a X: ya fue procesado en esta rama.
            P &= ~v_bit
            X |= v_bit

            # Actualizar candidates eliminando directamente v.
            # Es equivalente a recalcular P & ~adj_mask[best_pivot] porque
            # v ya no está en P, y el pivote no cambia en este ciclo.
            candidates &= ~v_bit

    # ── Llamada inicial ─────────────────────────────────────────────────────────
    # Al inicio: r_size = 0, P = todos los vértices, X = vacío.
    all_vertices = full_mask
    expand(r_size=0, P=all_vertices, X=0)

    return max_clique_size

<>:30: SyntaxWarning: invalid escape sequence '\ '
<>:30: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_4057/93218190.py:30: SyntaxWarning: invalid escape sequence '\ '
  reduciendo los candidatos a explorar a P \ N(u).


In [4]:
def hamming_distance(x: int, y: int, n: int) -> int:
    """
    Calcula la distancia de Hamming entre dos enteros x e y,
    considerando solo los n bits menos significativos.
    """
    return (x ^ y).bit_count()  # Python 3.8+: bit_count() es más rápido que bin().count()

def build_adjacency_mask(n: int, d: int) -> list:
    """
    Construye una lista de máscaras de adyacencia para el grafo H(n,d).
    - Cada vértice se representa por un entero de 0 a 2^n - 1.
    - adj_mask[i] es un entero cuyo bit j está a 1 si el vértice i es adyacente a j (j ≠ i y d_H(i,j) ≥ d).
    - Usamos un solo entero por vértice (Python int de precisión arbitraria), capaz de manejar hasta 2^n bits.
    """
    N = 1 << n                     # N = 2^n, número de vértices
    adj_mask = [0] * N             # Inicializar lista de máscaras

    # Precalculamos todas las distancias? Podríamos, pero O(N^2) es inevitable para construir el grafo.
    # Sin embargo, podemos acelerar usando la propiedad de que la distancia de Hamming es el número de unos en XOR.
    for i in range(N):
        # Para cada i, recorremos j > i y llenamos simétricamente
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) >= d:
                # Establecer el bit j en la máscara de i
                adj_mask[i] |= (1 << j)
                # Y el bit i en la máscara de j
                adj_mask[j] |= (1 << i)
    return adj_mask, N

In [11]:
n= 8
d= 4 
adj_mask, N = build_adjacency_mask(n,d)
result = bron_kerbosch_max_clique2(adj_mask, N)

In [12]:
result

16

In [ ]:
"""
Algoritmo optimizado para calcular A(n, d) (máximo tamaño de un código binario
de longitud n y distancia mínima d) usando búsqueda de conjunto independiente
máximo con poda por coloración y simetría de traslación.

Basado en los métodos descritos en:
- Östergård, Baicheva, Kolev (2000) - Optimal binary one-error-correcting codes
- Best (1980) - Binary codes with minimum distance four
- Tomita et al. (2003) - Algoritmo de clique máximo con poda por coloración

Representación: bitsets (enteros de Python) para velocidad.
"""

import sys
import time
from collections import deque

sys.setrecursionlimit(1000000)

def hamming_distance(x: int, y: int, n: int) -> int:
    """Distancia de Hamming entre dos enteros (n bits)."""
    return (x ^ y).bit_count()

def build_conflict_graph(n: int, d: int):
    """
    Construye el grafo de conflictos: vértices = palabras de n bits.
    Arista (i, j) si distancia(i, j) < d (es decir, NO pueden estar juntos en el código).
    Retorna:
        adj: lista de enteros (bitsets) de longitud N, donde adj[i] tiene bit j=1 si hay conflicto.
        N: número de vértices.
    """
    N = 1 << n
    adj = [0] * N
    # Precalculamos distancias solo para i < j y llenamos simétricamente
    # Para n=10, N=1024, bucles ~ 500k iteraciones, aceptable.
    for i in range(N):
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) < d:
                adj[i] |= (1 << j)
                adj[j] |= (1 << i)
    return adj, N

def max_independent_set(adj, N, use_translation=True):
    """
    Encuentra el tamaño del conjunto independiente máximo (MIS) en el grafo
    representado por bitsets de adyacencia (conflictos).
    Si use_translation=True, fija el vértice 0 en la solución (por traslación).
    """
    # Orden degenerado: vértices por grado ascendente (menos conflictos primero)
    degrees = [adj[i].bit_count() for i in range(N)]
    order = sorted(range(N), key=lambda v: degrees[v])
    # Reordenamos la matriz de adyacencia según ese orden
    # Para simplificar, trabajamos con índices originales pero usamos el orden
    # para la iteración inicial. El algoritmo de MIS con bitsets no requiere reordenar
    # si manejamos los bits correctamente. Lo haremos directamente.

    max_size = 0

    # Función de coloración greedy: devuelve el número de colores y una lista de vértices
    # ordenados por color descendente (para poda)
    def greedy_color(cand_bits):
        # cand_bits es un bitset de vértices candidatos
        # Extraemos lista de vértices
        vertices = []
        temp = cand_bits
        while temp:
            v = (temp & -temp).bit_length() - 1
            vertices.append(v)
            temp &= temp - 1
        if not vertices:
            return [], 0
        # Ordenar por grado descendente (dentro del subgrafo inducido) -> mejora coloración
        # El grado en el subgrafo es el número de vecinos dentro de cand_bits
        # Precomputamos para cada vértice su vecindario (ya lo tenemos en adj)
        vertices.sort(key=lambda v: (cand_bits & adj[v]).bit_count(), reverse=True)
        color = {}
        max_color = 0
        for v in vertices:
            # Colores usados por vecinos ya coloreados que están en cand_bits
            used = set()
            neigh = adj[v] & cand_bits
            ntemp = neigh
            while ntemp:
                u = (ntemp & -ntemp).bit_length() - 1
                if u in color:
                    used.add(color[u])
                ntemp &= ntemp - 1
            # Asignar el color más pequeño disponible
            c = 0
            while c in used:
                c += 1
            color[v] = c
            if c > max_color:
                max_color = c
        # Ordenar vértices por color descendente (los de mayor color primero)
        sorted_vertices = sorted(vertices, key=lambda v: color[v], reverse=True)
        return sorted_vertices, max_color

    # Búsqueda recursiva de MIS
    def expand(current_bits, cand_bits):
        nonlocal max_size
        # current_bits: vértices ya seleccionados (independientes)
        # cand_bits: vértices que no son adyacentes a current_bits (potenciales)
        current_size = current_bits.bit_count()
        # Poda por tamaño: si no se puede superar max_size
        if current_size + cand_bits.bit_count() <= max_size:
            return
        if cand_bits == 0:
            if current_size > max_size:
                max_size = current_size
            return

        # Coloración greedy para obtener cota superior
        sorted_cand, colors = greedy_color(cand_bits)
        if current_size + colors <= max_size:
            return

        # Recorrer los candidatos en el orden obtenido
        for v in sorted_cand:
            v_bit = 1 << v
            # Verificar que v sigue en cand_bits (puede haber sido eliminado en iteraciones anteriores)
            if not (cand_bits & v_bit):
                continue
            # Llamada recursiva: añadir v, nuevos candidatos = (cand_bits ∩ no_vecinos(v))
            # Los vecinos de v (conflictos) se eliminan de cand_bits
            new_cand = cand_bits & ~adj[v]
            expand(current_bits | v_bit, new_cand)
            # Eliminar v de cand_bits para futuras iteraciones
            cand_bits &= ~v_bit
            # Si después de eliminar, el tamaño máximo posible ya no mejora, romper
            if current_size + cand_bits.bit_count() <= max_size:
                break

    # Si usamos traslación, podemos fijar que el código contiene el 0.
    # Esto es válido porque cualquier código puede trasladarse (sumar una palabra)
    # para que contenga el cero. Luego el tamaño máximo no cambia.
    if use_translation:
        # El vértice 0 está en la solución. Entonces sus vecinos (conflictos) quedan excluidos.
        # Los candidatos iniciales son todos los vértices que no son vecinos de 0.
        # Además, el conjunto actual contiene solo 0.
        current = 1 << 0
        cand = ((1 << N) - 1) & ~adj[0] & ~(1 << 0)
        expand(current, cand)
    else:
        expand(0, (1 << N) - 1)

    return max_size

def A(n, d):
    """Calcula A(n, d) usando el algoritmo de MIS."""
    print(f"Construyendo grafo de conflictos para n={n}, d={d}...")
    adj, N = build_conflict_graph(n, d)
    print(f"Vértices: {N}")
    # Para d par, podemos restringirnos a palabras de peso par (por traslación y simetría)
    # Esto reduce el espacio a la mitad y acelera mucho.
    if d % 2 == 0 and n >= d:
        # Construir un subgrafo solo con palabras de peso par
        # Pero debemos mantener la compatibilidad con el algoritmo.
        # Una forma simple: filtrar los vértices que tienen peso impar.
        # Modificamos la matriz de adyacencia para que solo incluya esos vértices.
        # Sin embargo, el código ya es bastante eficiente; para n=10, esto ayuda.
        # Implementamos una versión opcional.
        even_vertices = [i for i in range(N) if i.bit_count() % 2 == 0]
        M = len(even_vertices)
        if M < N:
            print(f"Reduciendo a {M} vértices (solo peso par)")
            # Reindexar los vértices pares
            old_to_new = {v: idx for idx, v in enumerate(even_vertices)}
            new_adj = [0] * M
            for i, v in enumerate(even_vertices):
                # Vecinos de v que también sean pares
                neigh = adj[v]
                mask = 0
                for u in even_vertices:
                    if (neigh >> u) & 1:
                        mask |= (1 << old_to_new[u])
                new_adj[i] = mask
            # Llamar al MIS con estos nuevos índices
            return max_independent_set(new_adj, M, use_translation=True)
    return max_independent_set(adj, N, use_translation=True)

if __name__ == "__main__":
    # Pruebas
    tests = [(9,4)]
    for n, d in tests:
        start = time.time()
        res = A(n, d)
        elapsed = time.time() - start
        print(f"A({n},{d}) = {res} (tiempo: {elapsed:.2f} segundos)")
        print()

Construyendo grafo de conflictos para n=9, d=4...
Vértices: 512
Reduciendo a 256 vértices (solo peso par)


In [1]:
import sys
import time

# Aumentamos el límite de recursión (aunque la profundidad máxima es pequeña, por seguridad)
sys.setrecursionlimit(1000000)

def hamming_distance(x: int, y: int, n: int) -> int:
    """Distancia de Hamming entre dos enteros (n bits)."""
    return (x ^ y).bit_count()

def build_conflict_graph(n: int, d: int):
    """
    Construye el grafo de conflictos: vértices = palabras de n bits.
    Arista (i, j) si distancia(i, j) < d.
    Retorna:
        adj: lista de enteros (bitsets) de longitud N, donde adj[i] tiene bit j=1 si hay conflicto.
        N: número de vértices.
    """
    N = 1 << n
    adj = [0] * N
    for i in range(N):
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) < d:
                adj[i] |= (1 << j)
                adj[j] |= (1 << i)
    return adj, N

def max_independent_set(adj, N, use_translation=True):
    """
    Encuentra el tamaño del conjunto independiente máximo (MIS) en el grafo
    representado por bitsets de adyacencia (conflictos).
    Si use_translation=True, fija el vértice 0 en la solución (por traslación).
    """
    # Orden degenerado: grados de cada vértice
    degrees = [adj[i].bit_count() for i in range(N)]
    # No reordenamos los vértices, pero podemos usar el orden para la poda inicial
    # (la coloración greedy ya maneja el orden por grado)

    max_size = 0

    # Función de coloración greedy (Tomita)
    def greedy_color(cand_bits):
        # cand_bits: bitset de vértices candidatos
        # Devuelve (lista_vertices_ordenados, número_de_colores)
        vertices = []
        bits = cand_bits
        while bits:
            v = (bits & -bits).bit_length() - 1
            vertices.append(v)
            bits &= bits - 1
        if not vertices:
            return [], 0
        # Ordenar por grado descendente dentro del subgrafo inducido
        vertices.sort(key=lambda v: (cand_bits & adj[v]).bit_count(), reverse=True)
        color = {}
        max_c = 0
        for v in vertices:
            # colores usados por vecinos ya coloreados
            used = set()
            neigh = adj[v] & cand_bits
            nbits = neigh
            while nbits:
                u = (nbits & -nbits).bit_length() - 1
                if u in color:
                    used.add(color[u])
                nbits &= nbits - 1
            c = 0
            while c in used:
                c += 1
            color[v] = c
            if c > max_c:
                max_c = c
        # Ordenar por color descendente (mejor para poda)
        sorted_vertices = sorted(vertices, key=lambda v: color[v], reverse=True)
        return sorted_vertices, max_c

    # Búsqueda recursiva
    def expand(current_bits, cand_bits):
        nonlocal max_size
        cur_sz = current_bits.bit_count()
        # Poda por tamaño
        if cur_sz + cand_bits.bit_count() <= max_size:
            return
        if cand_bits == 0:
            if cur_sz > max_size:
                max_size = cur_sz
            return

        # Coloración para poda superior
        sorted_cand, colors = greedy_color(cand_bits)
        if cur_sz + colors <= max_size:
            return

        # Recorrer candidatos en el orden dado
        for v in sorted_cand:
            v_bit = 1 << v
            if not (cand_bits & v_bit):
                continue
            # Nuevos candidatos: eliminar v y todos sus vecinos
            new_cand = cand_bits & ~(adj[v] | v_bit)
            expand(current_bits | v_bit, new_cand)
            # Eliminar v de cand_bits para futuras iteraciones (rama donde no se toma v)
            cand_bits &= ~v_bit
            # Poda adicional: si lo que queda ya no puede mejorar, salir
            if cur_sz + cand_bits.bit_count() <= max_size:
                break

    # Si usamos traslación, fijamos el vértice 0 en la solución
    if use_translation:
        # El vértice 0 está en current, luego sus vecinos y él mismo no pueden estar en candidatos
        current = 1 << 0
        all_vertices = (1 << N) - 1
        cand = all_vertices & ~(adj[0] | (1 << 0))
        expand(current, cand)
    else:
        expand(0, (1 << N) - 1)

    return max_size

def A(n, d):
    """Calcula A(n,d) usando el algoritmo de MIS con poda por coloración."""
    print(f"Construyendo grafo de conflictos para n={n}, d={d}...")
    adj, N = build_conflict_graph(n, d)
    print(f"Vértices totales: {N}")

    # Si d es par, restringir a palabras de peso par (reduce el espacio a la mitad)
    if d % 2 == 0 and n >= d:
        even_vertices = [i for i in range(N) if i.bit_count() % 2 == 0]
        M = len(even_vertices)
        print(f"Reduciendo a {M} vértices (solo peso par)")
        # Reindexar
        old_to_new = {v: idx for idx, v in enumerate(even_vertices)}
        new_adj = [0] * M
        for i, v in enumerate(even_vertices):
            mask = 0
            neigh = adj[v]
            for u in even_vertices:
                if (neigh >> u) & 1:
                    mask |= (1 << old_to_new[u])
            new_adj[i] = mask
        # Ejecutar MIS sobre el grafo reducido
        result = max_independent_set(new_adj, M, use_translation=True)
        return result
    else:
        return max_independent_set(adj, N, use_translation=True)

if __name__ == "__main__":
    tests = [(8,3)]
    for n, d in tests:
        start = time.time()
        res = A(n, d)
        elapsed = time.time() - start
        print(f"A({n},{d}) = {res}  (tiempo: {elapsed:.2f} segundos)")
        print()

Construyendo grafo de conflictos para n=8, d=3...
Vértices totales: 256
A(8,3) = 17  (tiempo: 7.39 segundos)



In [2]:
import sys
import time

sys.setrecursionlimit(1000000)

def hamming_distance(x: int, y: int, n: int) -> int:
    return (x ^ y).bit_count()

def build_conflict_graph(n: int, d: int):
    N = 1 << n
    adj = [0] * N
    for i in range(N):
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) < d:
                adj[i] |= (1 << j)
                adj[j] |= (1 << i)
    return adj, N

def max_independent_set(adj, N, use_translation=True, verbose=False):
    max_size = 0
    # Grados para posible orden inicial (no utilizado directamente)
    # degrees = [adj[i].bit_count() for i in range(N)]

    # Coloración greedy (Tomita) con corrección del número de colores
    def greedy_color(cand_bits):
        # Extraer lista de vértices
        vertices = []
        bits = cand_bits
        while bits:
            v = (bits & -bits).bit_length() - 1
            vertices.append(v)
            bits &= bits - 1
        if not vertices:
            return [], 0
        # Ordenar por grado descendente dentro del subgrafo
        vertices.sort(key=lambda v: (cand_bits & adj[v]).bit_count(), reverse=True)
        color = {}
        max_color = -1
        for v in vertices:
            used = set()
            neigh = adj[v] & cand_bits
            nbits = neigh
            while nbits:
                u = (nbits & -nbits).bit_length() - 1
                if u in color:
                    used.add(color[u])
                nbits &= nbits - 1
            c = 0
            while c in used:
                c += 1
            color[v] = c
            if c > max_color:
                max_color = c
        num_colors = max_color + 1   # <--- CORRECCIÓN AQUÍ
        # Ordenar por color descendente
        sorted_vertices = sorted(vertices, key=lambda v: color[v], reverse=True)
        return sorted_vertices, num_colors

    # Búsqueda recursiva
    def expand(current_bits, cand_bits):
        nonlocal max_size
        cur_sz = current_bits.bit_count()
        # Poda por tamaño simple
        if cur_sz + cand_bits.bit_count() <= max_size:
            return
        if cand_bits == 0:
            if cur_sz > max_size:
                max_size = cur_sz
                if verbose:
                    print(f"Nueva clique de tamaño {max_size}")
            return

        # Obtener orden y cota por coloración
        sorted_cand, colors = greedy_color(cand_bits)
        if cur_sz + colors <= max_size:
            return

        # Iterar sobre candidatos
        for v in sorted_cand:
            v_bit = 1 << v
            if not (cand_bits & v_bit):
                continue
            # Nuevos candidatos: eliminar v y todos sus vecinos
            new_cand = cand_bits & ~(adj[v] | v_bit)
            expand(current_bits | v_bit, new_cand)
            # Eliminar v de cand_bits para futuras iteraciones
            cand_bits &= ~v_bit
            # Poda adicional
            if cur_sz + cand_bits.bit_count() <= max_size:
                break

    # Inicialización: fijar el vértice 0 si se usa traslación
    if use_translation:
        all_vertices = (1 << N) - 1
        # El vértice 0 está en la solución, luego sus vecinos y él mismo quedan fuera de candidatos
        current = 1 << 0
        cand = all_vertices & ~(adj[0] | (1 << 0))
        expand(current, cand)
    else:
        expand(0, (1 << N) - 1)

    return max_size

def A(n, d):
    print(f"Construyendo grafo de conflictos para n={n}, d={d}...")
    adj, N = build_conflict_graph(n, d)
    print(f"Vértices totales: {N}")

    # Para distancia par, restringir a peso par (reduce a la mitad)
    if d % 2 == 0 and n >= d:
        even_vertices = [i for i in range(N) if i.bit_count() % 2 == 0]
        M = len(even_vertices)
        print(f"Reduciendo a {M} vértices (solo peso par)")
        old_to_new = {v: idx for idx, v in enumerate(even_vertices)}
        new_adj = [0] * M
        for i, v in enumerate(even_vertices):
            mask = 0
            neigh = adj[v]
            for u in even_vertices:
                if (neigh >> u) & 1:
                    mask |= (1 << old_to_new[u])
            new_adj[i] = mask
        # Nota: en el grafo reducido, la traslación se aplica sobre índices nuevos
        # Sin embargo, el vértice que originalmente era 0 (si es par) se convierte en el índice correspondiente
        # Para mantener la simetría, seguimos usando traslación (fijamos el nuevo índice del 0)
        # En este caso, el 0 original (que tiene peso 0) siempre está en even_vertices, y su nuevo índice es old_to_new[0].
        # El algoritmo de MIS con traslación ya fija el vértice 0 (el nuevo índice 0). Pero cuidado: el nuevo índice 0 puede no corresponder al vértice 0 original.
        # La traslación es válida sobre el grafo reducido porque el código óptimo se puede trasladar dentro del subespacio de peso par? Esto requiere verificación.
        # Por simplicidad, llamamos a max_independent_set con use_translation=True; el vértice fijo será el de índice 0 en el nuevo grafo.
        # Para que sea correcto, debemos reindexar de modo que el nuevo vértice 0 sea el que tenía peso 0 original.
        # En even_vertices, el 0 original está en la posición 0 si hemos construido la lista en orden ascendente (por construcción, even_vertices está ordenada).
        # Por lo tanto, el nuevo índice 0 corresponde al vértice 0 original. Así que use_translation=True es correcto.
        result = max_independent_set(new_adj, M, use_translation=True)
        return result
    else:
        return max_independent_set(adj, N, use_translation=True)

if __name__ == "__main__":
    tests = [(8,3)]  # Primero probamos este; luego (9,4) y (10,4)
    for n, d in tests:
        start = time.time()
        res = A(n, d)
        elapsed = time.time() - start
        print(f"A({n},{d}) = {res}  (tiempo: {elapsed:.2f} segundos)")

Construyendo grafo de conflictos para n=8, d=3...
Vértices totales: 256
A(8,3) = 18  (tiempo: 9.77 segundos)


In [3]:
import sys
import time

sys.setrecursionlimit(1000000)

def hamming_distance(x: int, y: int, n: int) -> int:
    return (x ^ y).bit_count()

def build_adjacency_graph(n: int, d: int):
    """Grafo de adyacencia: arista si distancia >= d."""
    N = 1 << n
    adj = [0] * N
    for i in range(N):
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) >= d:
                adj[i] |= (1 << j)
                adj[j] |= (1 << i)
    return adj, N

def max_clique_tomita(adj, N, use_translation=True):
    """
    Algoritmo de Tomita (poda por coloración) para encontrar el tamaño de la clique máxima.
    """
    max_size = 0

    def greedy_color(cand_bits):
        # Devuelve (lista_de_vertices_ordenados, número_de_colores)
        vertices = []
        bits = cand_bits
        while bits:
            v = (bits & -bits).bit_length() - 1
            vertices.append(v)
            bits &= bits - 1
        if not vertices:
            return [], 0
        # Ordenar por grado descendente dentro del subgrafo
        vertices.sort(key=lambda v: (cand_bits & adj[v]).bit_count(), reverse=True)
        color = {}
        max_color = -1
        for v in vertices:
            used = set()
            neigh = adj[v] & cand_bits
            nbits = neigh
            while nbits:
                u = (nbits & -nbits).bit_length() - 1
                if u in color:
                    used.add(color[u])
                nbits &= nbits - 1
            c = 0
            while c in used:
                c += 1
            color[v] = c
            if c > max_color:
                max_color = c
        num_colors = max_color + 1
        # Ordenar por color descendente (mejor para poda)
        sorted_vertices = sorted(vertices, key=lambda v: color[v], reverse=True)
        return sorted_vertices, num_colors

    def expand(R_bits, P_bits, X_bits):
        nonlocal max_size
        # Poda por tamaño
        if R_bits.bit_count() + P_bits.bit_count() <= max_size:
            return
        if P_bits == 0 and X_bits == 0:
            size = R_bits.bit_count()
            if size > max_size:
                max_size = size
            return

        # Coloración para poda
        sorted_cand, colors = greedy_color(P_bits)
        if R_bits.bit_count() + colors <= max_size:
            return

        # Elegir pivote: vértice en P∪X con máximo grado en P
        # Simplificado: tomamos el primer vértice de P (suficientemente bueno)
        # Para mejor rendimiento se puede calcular el que maximiza |P ∩ N(u)|
        u = -1
        max_deg = -1
        union = P_bits | X_bits
        temp = union
        while temp:
            u_bit = temp & -temp
            uu = (u_bit).bit_length() - 1
            deg = (P_bits & adj[uu]).bit_count()
            if deg > max_deg:
                max_deg = deg
                u = uu
            temp ^= u_bit
        # Candidatos = P \ N(u) (los que no son vecinos de u)
        candidates = P_bits & ~adj[u]

        # Recorrer candidatos en el orden de sorted_cand (pero solo los que están en candidates)
        for v in sorted_cand:
            v_bit = 1 << v
            if not (candidates & v_bit):
                continue
            expand(R_bits | v_bit, P_bits & adj[v], X_bits & adj[v])
            P_bits &= ~v_bit
            X_bits |= v_bit
            if R_bits.bit_count() + P_bits.bit_count() <= max_size:
                break

    all_vertices = (1 << N) - 1
    if use_translation:
        # Fijamos que el vértice 0 pertenece a la clique (por traslación)
        # Entonces R = {0}, P = vecinos de 0, X = vacío
        R0 = 1 << 0
        P0 = all_vertices & adj[0]
        expand(R0, P0, 0)
    else:
        expand(0, all_vertices, 0)
    return max_size

def A(n, d):
    print(f"Construyendo grafo de adyacencia (distancia >= d) para n={n}, d={d}...")
    adj, N = build_adjacency_graph(n, d)
    print(f"Vértices: {N}")

    # Reducción por paridad: si d es par, restringir a palabras de peso par
    if d % 2 == 0 and n >= d:
        even_vertices = [i for i in range(N) if i.bit_count() % 2 == 0]
        M = len(even_vertices)
        print(f"Reduciendo a {M} vértices (solo peso par)")
        # Reindexar
        old_to_new = {v: idx for idx, v in enumerate(even_vertices)}
        new_adj = [0] * M
        for i, v in enumerate(even_vertices):
            mask = 0
            neigh = adj[v]
            for u in even_vertices:
                if (neigh >> u) & 1:
                    mask |= (1 << old_to_new[u])
            new_adj[i] = mask
        # En el subgrafo, la traslación se aplica reindexando el 0 original (que está en even_vertices)
        # El nuevo índice del 0 es old_to_new[0]; pero el algoritmo con traslación fija el vértice 0 (índice 0).
        # Para que funcione, debemos reordenar los índices de modo que el 0 original quede en la posición 0.
        # Como even_vertices está ordenada de menor a mayor, el 0 está en la primera posición si está incluido (lo está).
        # Por lo tanto, old_to_new[0] = 0. Así que use_translation=True es correcto.
        result = max_clique_tomita(new_adj, M, use_translation=True)
        return result
    else:
        return max_clique_tomita(adj, N, use_translation=True)

if __name__ == "__main__":
    tests = [(8,3), (9,4), (10,4)]
    for n, d in tests:
        start = time.time()
        res = A(n, d)
        elapsed = time.time() - start
        print(f"A({n},{d}) = {res}  (tiempo: {elapsed:.2f} segundos)")

Construyendo grafo de adyacencia (distancia >= d) para n=8, d=3...
Vértices: 256


KeyboardInterrupt: 

In [4]:
import sys
import time
from itertools import combinations

sys.setrecursionlimit(1000000)

def hamming_distance(x: int, y: int, n: int) -> int:
    return (x ^ y).bit_count()

def build_conflict_bitset(n: int, d: int):
    """Crea bitset de conflictos (1 si distancia < d)"""
    N = 1 << n
    conflict = [0] * N
    for i in range(N):
        for j in range(i + 1, N):
            if hamming_distance(i, j, n) < d:
                conflict[i] |= (1 << j)
                conflict[j] |= (1 << i)
    return conflict, N

def max_independent_set_backtrack(n, d):
    conflict, N = build_conflict_bitset(n, d)

    # Ordenamos los vértices por grado ascendente (menos conflictivos primero)
    # Esto mejora la poda y reduce ramas
    vertices = list(range(N))
    vertices.sort(key=lambda v: conflict[v].bit_count())
    # Reindexamos
    new_index = {v: i for i, v in enumerate(vertices)}
    M = N
    new_conflict = [0] * M
    for i, v in enumerate(vertices):
        c = 0
        for j, u in enumerate(vertices):
            if (conflict[v] >> u) & 1:
                c |= (1 << j)
        new_conflict[i] = c
    conflict = new_conflict
    all_vertices = (1 << M) - 1

    # ---------- Cotas superiores (Johnson) ----------
    def upper_bound_by_johnson(current_size, cand_bits):
        """Cota superior del número de vértices que se pueden añadir aún,
        usando la cota de Johnson para códigos de longitud n y distancia d.
        (Es una cota muy ajustada para d=3 y d=4)"""
        # Para simplificar, usamos el tamaño de cand_bits como cota superior trivial.
        # Pero podríamos implementar la cota de Johnson real.
        # En la práctica, para n<=10 la poda por coloración es más efectiva.
        return cand_bits.bit_count()

    # Mejor solución encontrada
    best = 0

    # Algoritmo de búsqueda recursiva con poda por coloración (en grafo de conflictos)
    def greedy_color(cand_bits):
        """Coloreo greedy del subgrafo inducido por cand_bits.
        Devuelve (lista_vertices_ordenados, número_de_colores)"""
        vertices = []
        bits = cand_bits
        while bits:
            v = (bits & -bits).bit_length() - 1
            vertices.append(v)
            bits &= bits - 1
        if not vertices:
            return [], 0
        # Ordenar por grado descendente dentro del subgrafo
        vertices.sort(key=lambda v: (cand_bits & conflict[v]).bit_count(), reverse=True)
        color = {}
        max_c = -1
        for v in vertices:
            used = set()
            neigh = conflict[v] & cand_bits
            nbits = neigh
            while nbits:
                u = (nbits & -nbits).bit_length() - 1
                if u in color:
                    used.add(color[u])
                nbits &= nbits - 1
            c = 0
            while c in used:
                c += 1
            color[v] = c
            if c > max_c:
                max_c = c
        num_colors = max_c + 1
        # Ordenar por color descendente
        sorted_vertices = sorted(vertices, key=lambda v: color[v], reverse=True)
        return sorted_vertices, num_colors

    def expand(current_bits, cand_bits):
        nonlocal best
        cur_size = current_bits.bit_count()
        # Poda por tamaño trivial
        if cur_size + cand_bits.bit_count() <= best:
            return
        if cand_bits == 0:
            if cur_size > best:
                best = cur_size
            return

        # Poda por coloración (en grafo de conflictos, el número cromático es cota superior del independiente máximo)
        sorted_cand, colors = greedy_color(cand_bits)
        if cur_size + colors <= best:
            return

        # Recorrer candidatos
        for v in sorted_cand:
            v_bit = 1 << v
            if not (cand_bits & v_bit):
                continue
            new_cand = cand_bits & ~(conflict[v] | v_bit)
            expand(current_bits | v_bit, new_cand)
            cand_bits &= ~v_bit
            if cur_size + cand_bits.bit_count() <= best:
                break

    # Si usamos traslación: fijamos el vértice 0 (que ahora es el índice del 0 original)
    # Encontramos el índice del 0 en el nuevo orden
    zero_idx = None
    for idx, v in enumerate(vertices):
        if v == 0:
            zero_idx = idx
            break
    if zero_idx is not None:
        current = 1 << zero_idx
        cand = all_vertices & ~conflict[zero_idx] & ~current
        expand(current, cand)
    else:
        expand(0, all_vertices)

    return best

def A(n, d):
    print(f"Calculando A({n},{d})...")
    start = time.time()
    res = max_independent_set_backtrack(n, d)
    elapsed = time.time() - start
    print(f"A({n},{d}) = {res}  (tiempo: {elapsed:.2f} segundos)")
    return res

if __name__ == "__main__":
    A(8, 3)
    # A(9, 4)
    # A(10, 4)

Calculando A(8,3)...
A(8,3) = 18  (tiempo: 10.20 segundos)


In [1]:
import subprocess
import tempfile
import os

In [2]:
def hamming_distance(x, y, n):
    return (x ^ y).bit_count()

def build_hamming_dimacs(n, d, filename):
    N = 1 << n
    edges = []
    # Recorremos solo i<j para evitar duplicados
    for i in range(N):
        for j in range(i+1, N):
            if hamming_distance(i, j, n) >= d:
                edges.append((i+1, j+1))   # DIMACS usa 1-indexado
    with open(filename, 'w') as f:
        f.write(f"p edge {N} {len(edges)}\n")
        for u, v in edges:
            f.write(f"e {u} {v}\n")

def A(n, d, cliquer_path="./cl"):
    # Crear archivo temporal con extensión .dimacs
    with tempfile.NamedTemporaryFile(mode='w', suffix='.dimacs', delete=False) as tmp:
        dimacs_file = tmp.name
    try:
        build_hamming_dimacs(n, d, dimacs_file)
        # Ejecutar cliquer con opciones: -u (unweighted), -w (solo peso), -r unweighted-coloring
        # Redirigimos stderr a /dev/null para limpiar la salida (opcional)
        result = subprocess.run(
            [cliquer_path, "-u", "-w", "-r", "unweighted-coloring", dimacs_file],
            capture_output=True, text=True, check=True
        )
        output = result.stdout.strip()
        # La salida puede ser "Largest clique: 20" o solo "20" (dependiendo de la versión)
        if "Largest clique:" in output:
            value = int(output.split(":")[1].strip())
        else:
            value = int(output)
        return value
    finally:
        os.unlink(dimacs_file)

if __name__ == "__main__":
    print("A(8,3) =", A(8,3))    # 20
    #print("A(9,4) =", A(9,4))    # 20
    #print("A(10,4) =", A(10,4))  # 40

FileNotFoundError: [Errno 2] No such file or directory: './cl'